In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import math
from math import atan2

import numpy as np # linear algebra
from matplotlib import animation
import matplotlib.pyplot as plt
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from IPython.display import display, HTML


In [ ]:
INPUT_DIR = "/kaggle/input/data-preprocessing"
RACE_PROGRAM_ID = ["program_number", "race_id"]

In [ ]:
def angle_between(v1, v2):
    dot = v1[0]*v2[0] + v1[1]*v2[1]      # dot product between [x1, y1] and [x2, y2]
    det = v1[0]*v2[1] - v1[1]*v2[0]      # determinant
    angle = atan2(det, dot)  # atan2(y, x) or atan2(sin, cos)
    if angle < 0:
        return (angle * 180 / np.pi) + 360
    return angle * 180 / np.pi

def angle_between(vel_x, vel_y, horse_x, horse_y):
    dot = vel_x * horse_x + vel_y * horse_y      # dot product between [x1, y1] and [x2, y2]
    det = vel_x * horse_y - vel_y * horse_x    # determinant
    angle = np.arctan2(det, dot)  # atan2(y, x) or atan2(sin, cos)
    
    angle = np.where(angle < 0, (angle * 180 / np.pi) + 360, angle * 180 / np.pi)

    return angle

def get_relative_position_table(tracking_df):
    
    # get direction df
    direction_df = tracking_df.copy()
    direction_df = direction_df.groupby(["track_id", "race_date", "race_number", "trakus_index"])[["latitude", "longitude"]].mean()
    direction_df = direction_df.add_prefix("ref_")
    direction_df[["vel_lat", "vel_long"]] = direction_df.diff(1).shift(-1).dropna()
    direction_df = direction_df.reset_index()
    
    # get relative position
    df = tracking_df.copy()
    df = df.merge(direction_df, on=["track_id", "race_date", "race_number", "trakus_index"])
    df["horse_relate_posit_lat"] = df["latitude"] - df["ref_latitude"]
    df["horse_relate_posit_long"] = df["longitude"] - df["ref_longitude"]

#     # get vector
#     df["velocity"] = df[["vel_long", "vel_lat"]].apply(tuple, axis=1)
#     df["horse_positon"] = df[["horse_relate_posit_long", "horse_relate_posit_lat"]].apply(tuple, axis=1)
    df["velocity"] = [*np.dstack((df.vel_long, df.vel_lat))[0]]
    df["horse_position"] = [*np.dstack((df.horse_relate_posit_long, df.horse_relate_posit_lat))[0]]
    
    # get angle
    df["angle"] = angle_between(
        df.vel_long.values,
        df.vel_lat.values, 
        df.horse_relate_posit_long.values, 
        df.horse_relate_posit_lat.values
    )
    # get relative distance
#     df["horse_relate_dist"] = df["horse_positon"].apply(lambda x: math.hypot(x[0], x[1]))
    df["horse_relate_dist"] = np.hypot(df.horse_relate_posit_long.values, df.horse_relate_posit_lat.values)
    return df

def get_zone(angle: pd.Series):
    return np.select(
        [
            angle.between(0, 45) | angle.between(315, 360),
            angle.between(135, 225),
            angle.between(45, 135),
            angle.between(225, 315)
        ],
        [
            "front",
            "back",
            "inner",
            "outter"
        ]
    )

In [ ]:
race = pd.read_csv(f"{INPUT_DIR}/race-updated.csv", parse_dates=["race_date"])
start = pd.read_csv(f"{INPUT_DIR}/start-updated.csv", parse_dates=["race_date"])
tracking = pd.read_csv(f"{INPUT_DIR}/tracking-updated.csv", parse_dates=["race_date"])
tracking = tracking.sort_values(RACE_PROGRAM_ID)

In [ ]:
finished_mask = ~tracking["is_finished"]
second_mask = tracking["trakus_index"].mod(4).eq(0)
tracking = tracking[finished_mask & second_mask]

In [ ]:
relative_posit_df = get_relative_position_table(tracking)

In [ ]:
relative_posit_df["zone"] = get_zone(relative_posit_df["angle"])
# relative_posit_df["cos_val"] = relative_posit_df[["horse_relate_dist", "angle"]].apply(tuple, axis=1).apply(lambda x: x[0] * np.cos(x[1] * np.pi / 180))
relative_posit_df["cos_val"] = relative_posit_df.horse_relate_dist.values * np.cos(relative_posit_df.angle.values * np.pi / 180)
relative_posit_df["rank"] = relative_posit_df.groupby(["race_id", "trakus_index"], as_index = False)[["cos_val"]].rank(ascending=False, method="dense")
relative_posit_df.head()

In [ ]:
def horse_relative_position_plot_zone(df):
    fig, ax = plt.subplots(figsize=(12,8))
    idxs = list(df.index)
    # Vector origin location
    X = [0]
    Y = [0]

    # Directional vectors
    U = df["velocity"][idxs[0]][0]
    V = df["velocity"][idxs[0]][1]


    # Creating plot
    plt.quiver(X, Y, U, V, color='r', units='xy', scale=3)
    c = {"back":"bo", "front":"go", "outter":"yo", "inner":"co"}
    j = 0 
    for i in idxs:
#         plt.quiver(X, Y, df["horse_positon"][i][0], df["horse_positon"][i][1], 
#                    units='xy', scale=1)
        plt.plot((0, df["horse_position"][i][0]), (0, df["horse_position"][i][1]), c[df["zone"][i]], linestyle="--")
        ax.text(df["horse_position"][i][0], df["horse_position"][i][1], df["program_number"][i], size=15)
        j += 1

    # plt.title('Single Vector')

    # x-lim and y-lim
    plt.xlim(-0.0002, 0.0002)
    plt.ylim(-0.0002, 0.0002)

    # Show plot with grid
    plt.grid()
    plt.show()

In [ ]:

def plot_by_trakus_index(data, steps):
    _df = data[(data["trakus_index"] == (4 * steps))]
    _df = _df.reset_index(drop=True)
    print(f'Race: {_df["race_id"][0]}, time {steps} s.')
    horse_relative_position_plot_zone(_df)
    display(_df[['program_number', 'velocity', 'horse_position', 'angle', 'horse_relate_dist', 'zone', 'cos_val', 'rank']].sort_values(by="angle"))

In [ ]:
race_df = relative_posit_df[relative_posit_df["race_id"] == '2019-07-17_7']

In [ ]:
plot_by_trakus_index(race_df, 20)

In [ ]:
idxs = [*race_df[(race_df["trakus_index"] == (4))].index]

In [ ]:
# %matplotlib widget

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
X = [0]
Y = [0]
ax.set_xlim(-0.0003, 0.0003)
ax.set_ylim(-0.0003, 0.0003)
ax.spines.right.set_visible(False)
ax.spines.top.set_visible(False)

n_horse = race_df.program_number.nunique()
n_trakus = race_df.trakus_index.nunique()
color = {"back":"blue", "front":"green", "outter":"yellow", "inner":"orange"}

lines = [plt.plot([], [])[0] for _ in range(n_horse)] #lines to animate

arrows = [plt.quiver(X, Y, [0], [0], color="r", units="xy", scale=4)] #arrows to animate
# texts = [plt.text(0, 0, "", size=15) for _ in range(n_horse)]

patches = lines + list(arrows) #things to animate

def init():
    #init lines
    for line in lines:
        line.set_data([], [])

    #init rectangles
    for arrow in arrows:
        arrow.set_UVC([0], [0])
        
#     for text in texts:
#         text.set_text(0, 0, "")

    return patches

def animate(i):
    _df = race_df[race_df["trakus_index"].eq(4 * (i + 1))]
    idxs = [*_df.index]
    U = _df["velocity"][idxs[0]][0]
    V = _df["velocity"][idxs[0]][1]
    #animate lines
    for j, line in enumerate(lines):
        current_program = idxs[j]
        line.set_data(
            (0, _df["horse_position"][current_program][0]), 
            (0, _df["horse_position"][current_program][1]),
        )
        line.set_color(color[_df["zone"][current_program]])
        line.set_linestyle("--")
        
#         text.set_text(df["horse_position"][i][0], df["horse_position"][i][1], df["program_number"][i], size=15)
    
    for arrow in arrows:
        arrow.set_UVC(U, V)

    return patches #return everything that must be updated

my_animation = animation.FuncAnimation(
    fig, 
    animate, 
    init_func=init,
    frames=n_trakus, 
    interval=500, 
    repeat=False,
    blit=True
)


In [ ]:
my_animation.save('myAnimation1.gif', writer='imagemagick', fps=30)

In [ ]:
HTML('<img src="./myAnimation1.gif" />')